# Stage8: exploratory metrics and frozen-backbone adapters
Private research run. Prior benchmarks have been observed; this run is not a fresh confirmatory test.

In [ ]:
import os,sys,subprocess,zipfile,io,base64,json,hashlib,shutil
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['HF_HUB_DISABLE_XET']='1'
work=Path('/kaggle/working')
archives=list(Path('/kaggle/input').rglob('baseline.zip'))
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
        z.extractall(work/'baseline')
else:
    manifests=[p for p in Path('/kaggle/input').rglob('human_manifest.json') if p.parent.name=='stage7']
    assert manifests, 'Private baseline dataset is attached but expected manifest is absent'
    shutil.copytree(manifests[0].parent.parent,work/'baseline',dirs_exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:z.extractall(work)
subprocess.run([sys.executable,'-m','pip','install','numpy==2.3.5','scipy==1.17.0','scikit-learn','pycocoevalcap==1.2','transformers==5.19.0','sentence-transformers==6.1.0','ftfy','regex','Pillow','pyarrow==25.0.1','gdown'],check=True)
import torch
assert torch.cuda.is_available(), 'CUDA required'
(work/'runtime.json').write_text(json.dumps({'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),'python':sys.version},indent=2))
subprocess.run([sys.executable,str(work/'stage8/verify.py'),'--baseline',str(work/'baseline')],check=True)
subprocess.run([sys.executable,str(work/'stage8/pipeline.py'),'--baseline',str(work/'baseline'),'--output',str(work/'stage8_results'),'--device','cuda','--gpu-benchmark'],check=True)
